<a href="https://colab.research.google.com/github/Eliezer-Carvalho/Adamastor-GPT/blob/main/Transformer%20-%20Attention%20Is%20All%20You%20Need.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Transformers**
<hr>

Os Transformers são uma arquitetura de Redes Neurais baseada no mecanismo <b> Attention</b>. <br>
Esta abordagem foi proposta em 2017 no artigo <b><i> ["Attention Is All You Need"](https://arxiv.org/pdf/1706.03762) </i></b> e desde então revolucionou o campo da Inteligência Artificial. <br>
Uma das maiores provas do impacto dos Transformers é o surgimento de modelos como: <br>

*    <b> [GPT](https://en.wikipedia.org/wiki/Generative_pre-trained_transformer) <i>(Generative Pre-trained Transformer)</i> </b>
*    <b> [BERT](https://en.wikipedia.org/wiki/BERT_(language_model)) <i>(Bidirectional Encoder Representations from Transformers)</i></b>









O objetivo deste Google Colab é explorar de forma prática a arquitetura dos Transformers. <br>
Ao longo do notebook, iremos percorrer as suas principais componentes construindo gradualmente a intuição por trás do modelo. <br>



---



In [9]:
from tokenizers import Tokenizer
from tokenizers.models import Unigram, BPE
from tokenizers.trainers import UnigramTrainer, BpeTrainer
from tokenizers.pre_tokenizers import Metaspace
import matplotlib.pyplot as plt



In [1]:
!wget https://raw.githubusercontent.com/Eliezer-Carvalho/Adamastor-GPT/refs/heads/main/data/Dataset.txt

--2026-05-12 00:14:13--  https://raw.githubusercontent.com/Eliezer-Carvalho/Adamastor-GPT/refs/heads/main/data/Dataset.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 800178 (781K) [text/plain]
Saving to: ‘Dataset.txt’

Dataset.txt         100%[===================>] 781.42K  --.-KB/s    in 0.05s   

2026-05-12 00:14:13 (14.5 MB/s) - ‘Dataset.txt’ saved [800178/800178]



### <b> 1. Tokenização </b>
Os Transformers consistem em operações sobre números. <br>
Para tal, deve haver primeiro um mapeamento de qualquer texto de entrada para representação numérica. <br>

A <b> Tokenização </b> divide o texto em pequenos fragmentos chamados <b> tokens</b>. <br>
Um token pode ser uma palavra inteira, parte de uma palavra ou até mesmo um sinal de pontuação. <br>

A forma como essa divisão é feita depende principalmente dos **[Algoritmos de Tokenização](https://huggingface.co/docs/transformers/tokenizer_summary)**. <br>
Entre os mais comuns estão:

*   **BPE (*Byte Pair Encoding*)**
*   **Unigram**
*   **SentencePiece**
*   **WordPiece**
*   **Word-Level Tokenization**
*   **Character-Level Tokenization**



Diferentes algoritmos produzem diferentes tokenizações, o que pode impactar o desempenho do modelo. <br>

Existem também os <b> [Pré Tokenizers](https://huggingface.co/docs/tokenizers/api/pre-tokenizers) </b> que aplicam uma transformação especial no dataset antes de aplicar o Algoritmo de Tokenização. <br>
<b> Pela minha experiência, nem sempre um Pré Tokenizer traz vantagens. </b> <br>


#### <b> Análise do Dataset </b>

In [10]:
import re   #https://pypi.org/project/regex/

with open ("Dataset.txt", "r", encoding = "utf-8") as f:
  dataset = f.read()

#https://regexbox.com/cheatsheet
print (f"Caractéres --> {len(dataset)}")
print (f"Palavras --> {len(re.findall(r'\w+', dataset))}")
print (f"Linhas --> {len(re.findall(r'\n', dataset))}")

Caractéres --> 784702
Palavras --> 141789
Linhas --> 24145


#### <b> Tokenizer </b>

<b> Neste projeto, criámos o nosso próprio Tokenizer de raiz! </b> <br>
Não é algo obrigatório, é possível usar outros Tokenizers já existentes.

In [11]:
alg_token = Tokenizer(Unigram())

trainer = UnigramTrainer()

alg_token.pre_tokenizer = Metaspace()

alg_token.train (["Dataset.txt"], trainer)

alg_token.save ("teste2")


In [14]:
tokenizer = Tokenizer.from_file ("teste")

In [17]:
from tokenizers.decoders import Metaspace

dataset2 = tokenizer.encode(dataset)
decoder = Metaspace()



print (tokenizer)
print (dataset2.tokens [:50])
print (dataset2.ids[:50])
print (decoder.decode(dataset2.tokens [:50]))

TypeError: object of type 'tokenizers.Tokenizer' has no len()

In [ ]:
alg_tokenização = Tokenizer(Unigram()) #Algoritmo de Tokenização

trainer = UnigramTrainer ( #Trainer
    vocab_size = 28000 #Representa o número de tokens únicos no dataset
)

#Sem Pré Tokenizer. #Foi realizado um estudo e foi decidida que a melhor abordagem seria um Tokenizer como Unigram. #Como já foi abordado, consoante o dataset, diferentes Algoritmos de Tokenização podem ser aplicados para ter melhor perfomance
alg_tokenização.train (["Dataset.txt"], trainer) #Treino

alg_tokenização.save("Unigram - 28000") #Guardar o modelo! Para poder usar em qualquer projeto

tokenizer = Tokenizer.from_file("/content/Unigram - 28000") #Upload do Tokenizer

dataset = tokenizer.encode (dataset) #Encode do dataset

#### **Visualização do Tokenizer**



*   **Tokens** -> unidades fundamentais de texto que representam <i> palavras</i>, <i>partes de palavras</i> ou <i>caracteres</i>.
*   **Ids** -> representação numérica dos tokens



In [ ]:
print (f"Estatísticas do Tokenizer --> {dataset}")
print (f"50 Primeiros Tokens --> {dataset.tokens[0:50]}")
print (f"50 Primeiros Ids --> {dataset.ids[0:50]}")

#### **Codificador e Descodificador de um Tokenizer**


In [ ]:
teste = tokenizer.encode ("Transformers são fixes e os Lusíadas são épicos!")

print ("Codificador:")
print (teste.tokens)
print (teste.ids, "\n")

print ("Descodificador:")
print (tokenizer.decode(teste.ids))


decoder = Fuse()
decoder.decode (teste.tokens)

#### **Espaço Dimensional de um Tokenizer**


In [ ]:
x = dataset.tokens[200:225]
y = dataset.ids[200:225]

plt.figure (figsize = (30, 10), dpi = 80)
plt.scatter (x, y, marker = "x", color = 'blue', s = 500)


for i in range (len(y)):

    plt.text(x[i], y[i] + 1000, y[i], ha = "center", size = 8)

plt.xticks (size = 12)
plt.xlabel ("Tokens", size = 18, labelpad = 20)

plt.yticks (list(range(0, 28000, 5000)), size = 15)
plt.ylabel ("ID's", size = 18, labelpad = 20)

plt.title ("Espaço Dimensional de um Tokenizer [200:225]", size = 22, pad = 20)
plt.show()

#### **Como avaliar um Tokenizer ?**

Consoante a tarefa, ou o tipo de dataset, podemos precisar de Tokenizers com técnicas de Tokenização diferentes. <br>
No caso deste projeto, o Português abordado nos Lusíadas, não é o <b>Português tradicional</b>. <br>

Por tal motivo, um Tokenizer comum, como <i>gpt</i> ou <i>bert</i>, não é a melhor abordagem. <br>
Durante o processo de criação do meu Tokenizer, pensei <b>"Como sei que o meu Tokenizer funciona ?"</b>

**[Fertility](https://arxiv.org/pdf/2310.08754)** é definida como o número médio de tokens necessários para representar uma palavra. <br>
Pode ser representada matemáticamente como:

$$
\frac {\text{Número de Tokens}} {\text{Número de Palavras no Dataset}}
$$
<br>
No meu caso, analisei vários tipos de Algoritmos de Tokenização <b>com</b> e <b>sem</b> Pré-Tokenizers, e cheguei à conclusão que o melhor método seria usar o Algoritmo de Tokenização <b>Unigram sem Pré-Tokenizers</b>. <br>
Após esta análise, estudei qual o melhor número de <b>vocab_size</b> para o meu dataset.

In [ ]:
tokenizer_fert = { #Dados obtidos calculando Fertility com vários tamanhos de vocabulário
    10000: 1.30, #Vocab_Size: Fertility
    15000: 1.19,
    20000: 1.16,
    25000: 1.12,
    30000: 1.12,
    35000: 1.12
}

plt.figure (figsize = (30, 10), dpi = 80)

plt.plot (list(tokenizer_fert.keys()),
          list(tokenizer_fert.values()),
          marker = 'o',
          markersize = 10,
          color = 'green'
          )

plt.xticks (size = 15)
plt.xlabel ("Tamanho do Vocabulário", size = 18, labelpad = 20)

plt.yticks (size = 15)
plt.ylabel ("Fertility", size = 18, labelpad = 20)

plt.title ("Fertility - Tokenizer Unigram", size = 22, pad = 20)
plt.show()



###


---





### **2. Tensor Shape**

Após a Tokenização, o dataset já se encontra convertido para uma representação numérica. <br>
A organização dos dados de entrada é feita através da definição de dois parâmetros fundamentais: <br>


1.   **Batch Size** - número de sequências processadas em paralelo durante o treino.
2.   **Sequence Length** - número de tokens considerados em cada sequência.


Esta abordagem é necessária porque os Transformers não processam todo o texto de uma vez, mas sim em blocos de tamanho fixo. <br>
De forma análoga à leitura humana, não é possível analisar um texto completo simultaneamente, é necessário dividi-lo em partes para conseguir identificar padrões e relações.

<b> Assim, os dados são representados sob a forma de um tensor com dimensão (B, T). </b>



In [ ]:
import torch #https://docs.pytorch.org/docs/stable/index.html
import torch.nn as nn #https://docs.pytorch.org/docs/stable/nn.html
import torch.nn.functional as F #https://docs.pytorch.org/docs/stable/nn.functional.html#torch-nn-functional

#### **Definição do Tensor e Dados de Treino vs Dados de Teste**

In [ ]:
tensor = torch.tensor (dataset.ids, dtype = torch.long)
print (len(tensor))
print (tensor [:100])

#Divisão de Dados de Treino e Dados de Teste
n = int (0.9 * len(tensor))
dados_treino = tensor [:n] #90% dos dados
dados_teste = tensor [n:] #10% dos dados


#### **Tensor (B, T)**

In [ ]:
Batch_Size = 4
Sequence_Length = 8
torch.manual_seed (1000)

posição = torch.randint (0, len (dados_treino) - Sequence_Length, (Batch_Size, ))
x = torch.stack ([dados_treino [i:i + Sequence_Length] for i in posição])
y = torch.stack ([dados_treino [i + 1: i + Sequence_Length + 1] for i in posição])

print (x.shape) #(B, T)
print (x)
print (y.shape) #(B, T)
print (y)

#mais para a frente
sda = x

###


---



### <b> 3. Embedding </b>

Word Embeddings são representações vetoriais de palavras.

Normalmente, cada token é mapeado para um <b> vetor de valores reais</b>, onde a sua posição no espaço vetorial reflete o seu significado. Palavras com significados semelhantes tendem a estar próximas nesse espaço, permitindo ao modelo capturar relações semânticas entre elas.

Neste ponto, introduz-se uma das variáveis fundamentais de uma arquitetura Transformer:


1.   <b> Embedding Dimension </b> - Define a dimensão dos vetores utilizados para representar cada token e é um dos principais fatores no número total de parâmetros do modelo.

Os Embeddings são parâmetros treináveis, ou seja, são ajustados durante o treino do modelo de forma a melhorar a representação dos tokens no espaço vetorial.

<b> Após a aplicação dos Embeddings passamos a ter um tensor com dimensão (B, T, C). </b>

In [ ]:
Embedding_Dimension = 16

Embedding = nn.Embedding (tokenizer.get_vocab_size(), Embedding_Dimension) #Porquê vocab_size ? Vocab_Size dá-nos o número dos tokens únicos que existem.
#Se usássemos por exemplo len(Tokenizer.tokens) não era correto porque estariamos a criar um vetor para tokens repetidos.
x_embedding = Embedding (x)
print (x_embedding.shape) #(B,T,C)
print (f"O Token {x[0,1]} agora é representado pelo vetor:")
print (x_embedding[0,1]) #Representação vetorial do primeiro token após embedding

#### **Representação dos Tokens e dos seus respetivos Embeddings**
Cada token passa a ser representado por um vetor.

Esta tabela também é excelente para perceber o que <b> nn.Embedding </b> faz por trás.

In [ ]:
import pandas as pd

dados = []

for i in range (8):
  dados.append ({
      "Token": sda[0,i],
      "Pós Embedding -> Sequence Length [0]": x_embedding[0,i].detach().cpu().numpy()
  })

df = pd.DataFrame (dados)
print(df.to_string()) #Melhor maneira de mostrar a tabela completa

###


---


### **4. Positional Encoding**

<b>Positional Encoding</b> é uma técnica que adiciona informação sobre a posição de cada token na sequência.

Isto é necessário porque os Transformers, por si só, não processam os tokens de forma sequencial (como as RNNs) e, por isso, não têm noção da ordem das palavras.

Ao introduzir esta informação, o modelo consegue distinguir entre a posição relativa ou absoluta dos tokens, o que é essencial para compreender a estrutura de uma frase e o significado do texto.

A intuição é a seguinte:

* “Rei ama rainha” ≠ “Rainha ama rei” — <b>a ordem importa</b>!

No paper original são utilizados encodings fixos. <br>
No entanto, em muitas implementações modernas, são utilizados positional encodings treináveis, que permitem ao modelo aprender a melhor representação para cada posição durante o treino.

**Neste projeto foram utilizados encodings treináveis.**

In [ ]:
Positional_Embedding = nn.Embedding (Sequence_Length, Embedding_Dimension) #Positinal Encoding não é nada mais do que um Embedding
Positional_Encoding = Positional_Embedding (torch.arange (Sequence_Length))
print (Positional_Encoding.shape)

###


---


### **5. Embedding + Positional Encoding**

Neste último passo, antes de introduzirmos o mecanismo <b> Attention</b>, é necessário combinar os Embeddings dos tokens com os Positional Encodings.

Esta combinação é feita através de uma soma elemento a elemento, permitindo que cada token transporte simultaneamente informação sobre o seu significado e a sua posição na sequência.

Este passo é fundamental, pois prepara a representação final que será utilizada pelo mecanismo <b> Attention<b>. <br>
Por essa razão, neste Notebook é apresentado como uma etapa separada e de especial importância no fluxo da arquitetura.

In [ ]:
x = x_embedding + Positional_Encoding #Dados que vão entrar no modelo
print (x.shape)
print (x[:100])

##### **Embedding + Positional Encoding**

In [ ]:
dados = []

for i in range (8):
  dados.append ({
      "Token": sda[0,i],
      "Embedding + Positional Encoding -> Sequence Length [0]": x[0,i].detach().cpu().numpy()
  })

df = pd.DataFrame (dados)
print(df.to_string()) #Melhor maneira de mostrar a tabela completa

###


---

### **6. [Attention](https://medium.com/@nitinmittapally/understanding-attention-in-transformers-a-visual-guide-df416bfe495a)**

<b> Attention </b> é um mecanismo que permite ao modelo focar-se em diferentes partes da sequência de entrada ao processar cada token.

De forma análoga à atenção humana, ao ler uma frase não analisamos cada palavra isoladamente, em vez disso, consideramos o contexto relevante para compreender o significado. <br>
Da mesma forma, o mecanismo <b> Attention </b> permite ao modelo atribuir diferentes níveis de importância aos vários tokens da sequência, capturando relações e dependências entre eles. <br>
Ao contrário das RNNs, este mecanismo permite acesso direto a todos os Tokens anteriores, facilitando a captura de dependências de longo alcance.

Este é o mecanismo central da arquitetura Transformer e pode ser aplicado de diferentes formas, destacando-se três variantes principais:

*   <b> Self-Attention </b> - cada token atende a todos os outros tokens da mesma sequência, permitindo capturar relações internas.
*   <b> Masked Self-Attention </b> - semelhante ao Self-Attention, mas com uma máscara que impede o modelo de “ver o futuro”, sendo essencial para tarefas de geração de texto.
*   <b> Cross-Attention </b> - permite relacionar duas sequências diferentes, onde uma sequência atende à outra


<b> Neste Notebook será utilizada a variante Masked Self-Attention, característica de modelos do tipo GPT. </b>

#### <b> Query, Key e Value (Q, K, V) </b>

<b> Attention </b> baseia-se em três representações vetoriais:

* <b> Query (Q) </b> - O que ando à procura ?
* <b> Key (K) </b> - O que represento ?
* <b> Value (V) </b> - Que informação trago comigo ?

A Attention é calculada através da comparação entre <b> Queries </b> e <b> Keys</b>, permitindo determinar o grau de relevância entre tokens. <br>
Esses pesos são depois aplicados aos <b> Values </b> para produzir a saída final. <br>
<b> Vamos ver em código: </b>

In [ ]:
B,T,C = x.shape #Batch, Sequence Length, Embedding_Dimension
#print (x.shape)
Head_Size = C = 16 #Head_Size representa a divisão do vetor #Se tiver um vetor com Emb_Dimension de valor 16, e tiver uma arquitetura Multi-Head Attention com 2 Heads, a minha Head_Size é 8. #Isto significa que por cada Head eu vou dar 8 valores do meu vetor de Embeddings

#C = Embedding_Dimension #Aplica uma Transformação Linear a partir dos Embeddings #Uma projeção desses Embeddings num espaço
Query = nn.Linear (C, Head_Size, bias = False)
Key = nn.Linear (C, Head_Size, bias = False)
Value = nn.Linear (C, Head_Size, bias = False)

#Aplico a projeção nos dados
Q = Query (x) #Y = X * Wq
K = Key (x) #Y = X * Wk
V = Value (x) #Y = X * Wv

print (x.shape)
print (Q.shape)
print (K.shape)
print (V.shape)

#### **Heatmap da Representação Vetorial do Embedding e dos respetivos Q, K e V**

In [ ]:
import seaborn as sns

df = pd.DataFrame({
    "Embedding": x [0,1].detach().numpy(), #x[0,1] vai selecionar um tensor da posição 1 da linha 0 do tensor
    "Query - O que ando à procura": Q [0,1].detach().numpy(),
    "Key - O que represento": K [0,1].detach().numpy(),
    "Value - Que informação trago comigo": V [0,1].detach().numpy(),
})

plt.figure (figsize = (25, 10))
sns.heatmap(df, cmap = "YlGnBu", annot = True, fmt = ".7f", annot_kws = {"size": 12})

plt.xticks (size = 14)
plt.yticks (size = 14)

plt.title(f"Embedding do Token {sda[0,1]} -> Q, K, V", size = 20, pad = 20)
plt.show()


#### **Representação do Espaço Dimensional do Embedding e dos respetivos Q, K e V**

In [ ]:
plt.figure (figsize = (30, 10))
sns.violinplot(df, cmap = "YlGnBu")

plt.xticks (size = 14)
plt.yticks (size = 14)

plt.title(f"Espaço Dimensional do Embedding do Token {sda[0,1]} -> Q, K e V", size = 20, pad = 20)
plt.show()

#### **Scaled Dot-Product Attention (Q @ K)**

**Scaled Dot-Product Attention** é o mecanismo central que permite aos tokens "interagir entre si" dentro de um Transformer.

Em vez de tratar cada posição isoladamente, o modelo passa a construir significado com base no contexto global.

É aqui que deixa de haver apenas transformações individuais (Q, K, V) e passa a haver comunicação entre tokens.

Um ponto importante é que este mecanismo é diferenciável e altamente <b> paralelizável </b>, o que o torna eficiente para treino em larga escala. <br>
Além disso, como todos os tokens "conversam" entre si simultaneamente, o modelo consegue capturar dependências de longo alcance — algo que era difícil em arquiteturas anteriores como RNNs.

In [ ]:
attention_scores = Q @ K.transpose (-2, -1) * (Head_Size ** -0.5) #(B, T, C) @ (B ,C ,T) --> (B, T, T) #Estamos a fazer com que os tokens falem entre si, troquem a sua informação
print (f"Após os Tokens falarem entre si (Q @ K.t) ->\n{attention_scores.shape}") # (B, T, T) porque são apenas relações entre os tokens
print (attention_scores[0])

tril = torch.tril (torch.ones(T, T)) #Cria uma máscara
attention_scores  = attention_scores.masked_fill (tril == 0, float ('-inf')) #Aplica a máscara na Atenção
print (f"\nAplicação de Mask Attention -> \n {attention_scores[0]} \n")

attention_scores = F.softmax (attention_scores, dim = -1) #Softmax para normalizar #A soma dos valores dá 1
print (f"Aplicação de Softmax para normalizar os valores -> \n {attention_scores[0]}\n")

output = attention_scores @ V #Resultado final mostra as relações entre todos os Tokens de uma Sequence length
print (f"Output (attention_scores @ V) -> \n{output.shape} \n{output[0]}")

###


---

### **7. [Add & Norm](https://arxiv.org/pdf/1607.06450)**

<b> Layer Normalization (LayerNorm ou Norm) </b> é uma técnica que estabiliza e acelera o treino de Redes Neurais. <br>
Normaliza as ativações dentro de cada exemplo (ao longo das features), garantindo média próxima de 0 e variância próxima de 1.

Ao fazê-lo, reduz a sensibilidade à escala dos dados e à inicialização dos pesos, tornando o treino mais estável e eficiente. <br>
É especialmente eficaz em <b> RNNs e Transformers </b>, onde ajuda a mitigar problemas de gradientes instáveis e melhora a convergência.

<b> Residual Connection (ou Add) </b> é uma componente simples mas fundamental em arquiteturas profundas.
Consiste em somar a entrada original ao output de uma camada.

Este mecanismo cria um <b> "um atalho" </b> para a informação, permitindo que o modelo preserve o sinal original ao longo das camadas.

Como resultado, facilita o fluxo de gradientes, reduz o problema de <i> vanishing gradients </i> e permite treinar redes muito mais profundas.

In [ ]:
LayerNorm = nn.LayerNorm (Embedding_Dimension) #Layer Norm

output = x + LayerNorm (output) #Residual Connection

print ("Output após Add & Norm ->")
print (output.shape)
print (output[0])

###


---

### **8. Feed Forward (Multi Layer Perceptron)**

Self Attention permite que os tokens recolham informação contextual ao longo de toda a sequência. Mas a Attention, por si só, é limitada pois é uma operação fundamentalmente linear.

Uma <b> Feedforward Neural Network (FNN ou Multi Layer Perceptron) </b> fornece exatamente este ingrediente em falta.

Tem como principal função aplicar uma <b> transformação não linear </b> à representação de cada token de forma independente. Como se fosse uma camada de <i> «pensamento» </i> do Transformer.

Depois de a Attention ter encaminhado a informação entre posições e montado representações sensíveis ao contexto, a MLP processa cada uma dessas representações através de uma pequena Rede Neural capaz de aprender limites de decisão curvos e de alta dimensão.

<b> É aqui que ocorre o verdadeiro trabalho computacional da compreensão da linguagem. </b>



In [ ]:
MLP = nn.Sequential (
    nn.Linear (Embedding_Dimension, 4 * Embedding_Dimension),
    nn.ReLU (),
    nn.Linear (Embedding_Dimension * 4, Embedding_Dimension)
)

output = MLP (output)
print ("Output após Feed Forward ->")
print (output.shape)
print (output[0])

###


---

### **9. Add & Norm (Sim, outra vez)**



In [ ]:
LayerNorm = nn.LayerNorm (Embedding_Dimension) #Layer Norm

output = x + LayerNorm (output) #Residual Connection

print ("Output após Add & Norm ->")
print (output.shape)
print (output[0])

###


---

### **10. Language Modelling**

Na tarefa de <b> Language Modelling </b>, a última camada transforma o estado escondido do modelo numa distribuição de probabilidades sobre o vocabulário.

Para isso, o modelo aplica primeiro uma projeção linear ao vetor de features, convertendo-o num vetor <b> (logits)</b> com dimensão igual ao número de palavras possíveis (vocab_size). <br>

De seguida, aplica a função que normaliza os dados, <b> Softmax</b>, que transforma esses valores em probabilidades.

O resultado é um vetor onde cada posição corresponde à <b> probabilidade</b> de uma determinada palavra ser o próximo token na sequência, permitindo ao modelo fazer previsões informadas sobre o que vem a seguir.

In [ ]:
Language_Modelling = nn.Linear (Embedding_Dimension, tokenizer.get_vocab_size())

logits = Language_Modelling (output)

probs = F.softmax (logits, dim = -1)

print ("Logits ->")
print (logits.shape)
print (logits[0])

print ("\nProbabilidades ->")
print (probs.shape)
print (probs[0])

###


---

# **Conclusão**


---


Este Google Colab teve como principal objetivo implementar e compreender a arquitetura Transformer, proposta no paper <i> <b> Attention Is All You Need</b></i>. <br>
Para além da implementação, serviu como uma base prática de aprendizagem, permitindo consolidar conceitos fundamentais.

Ao longo do projeto, foram introduzidos e explorados vários conceitos essenciais, tais como:
<b>
- Sequence Length
- Batch Size
- Embedding Dimension
- Head Size
- Number of Heads
- Logits
- Probabilidades
</b>

E percorremos estrututas cruciais dos Transformers tais como:
<b>
- Tokenization
- Tensor Shape
- Embedding
- Positional Encoding
- Embedding + Positional Encoding
- Attention
- Add & Norm
- Feed Forward
- Add & Norm
- Language Modelling
</b>

Estes elementos não são apenas detalhes técnicos, mas sim componentes estruturais que influenciam diretamente o comportamento, a eficiência e a capacidade de generalização do modelo.


No seu conjunto, estes conceitos formam a base do funcionamento dos <b> LLM's</b> e estão presentes de forma recorrente no desenvolvimento e aplicação de modelos modernos de <b> Processamento de Linguagem Natural</b>. <br>

<b> A compreensão destes elementos é, portanto, fundamental para evoluir de uma perspetiva introdutória para uma utilização mais avançada e crítica desta arquitetura. </b>